# Notebook: Applied Stochastic Calculus

ใช้ Python 3 และ standard library แล้วเลือก **Run All** ไม่ต้องติดตั้งแพ็กเกจหรือดาวน์โหลดข้อมูล คำอธิบายอ่านจาก `applied-stochastic-calculus.md` พร้อมการทดลองสมมติที่คำนวณขึ้นใหม่ ทุกการสุ่มระบุ seed; Notebook ใช้ `random.gauss` จึงเป็นการทดลองที่ทำซ้ำได้แยกจากตัวสุ่มบนเว็บไซต์ ตัวอย่าง Monte Carlo มี sampling error และ finite difference มี discretization error การผ่าน checks ไม่ใช่บทพิสูจน์การลู่เข้า

[เปิดบทเรียน](https://nutdnuy.github.io/quantitative-finance-notes/applied-stochastic-calculus.html)

# Applied Stochastic Calculus

เมื่อสิ่งที่เราสนใจเป็นฟังก์ชันของตัวแปรสุ่ม เราจะหาการเปลี่ยนแปลงของมันอย่างไร

> “ตรรกะและสัญชาตญาณต่างมีบทบาทที่จำเป็น ทั้งสองสิ่งขาดไปไม่ได้”
>
> <span lang="en">“Thus logic and intuition have each their necessary rôle. Each is indispensable.”</span>
>
> — **Henri Poincaré** · [*The Value of Science, บท I §V*](https://www.gutenberg.org/cache/epub/39713/pg39713-images.html)

ในบทนี้เราจะเข้าใจ Stochastic Calculus อย่าพึงถอดใจเมื่อเจอสิ่งที่ยาก Stochastic  แปลว่าสุ่ม Calculus คือการศึกษาการเปลี่ยนแปลง เรเพียงแต่จะศึกษาการเปลี่ยนแปลงของระบบที่สุ่ม

บท [Transition Density Functions](https://nutdnuy.github.io/quantitative-finance-notes/transition-density-functions.html) ตอบว่า “จากจุดเริ่มต้นหนึ่ง อนาคตกระจายไปอย่างไร” แต่ในการเงิน เรามักไม่ได้สนใจตัวแปรนั้นเพียงตัวเดียว เราอาจสนใจ log ของราคา ราคายกกำลังสอง หรือมูลค่าสัญญาที่ขึ้นกับราคาและเวลา

ถ้า S เปลี่ยนแบบสุ่ม แล้ว F(S,t) จะเปลี่ยนตามกฎใด? คำตอบคือ **[Itô’s lemma](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#itos-lemma)** ซึ่งเป็นกฎลูกโซ่สำหรับกระบวนการแบบ Itô และเป็นเครื่องมือหลักของบทนี้

In [1]:
import math
import random
def close(actual, expected, tolerance=1e-10):
    assert math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance), (actual, expected)
print("Python standard library only. Each Monte Carlo experiment declares its own seed.")
print("Notebook and website use separate random-number implementations; their simulated paths need not match.")

Python standard library only. Each Monte Carlo experiment declares its own seed.
Notebook and website use separate random-number implementations; their simulated paths need not match.


## 1. time step ย่อครึ่งหนึ่ง แต่ขนาดช็อกไม่ได้ย่อครึ่งหนึ่ง

เริ่มจากเหรียญยุติธรรมและผลการโยนที่เป็นอิสระ ให้ Rᵢ เป็น +1 หรือ −1 ด้วยโอกาสอย่างละ 1/2 และ Cₙ เป็นยอดสะสมหลัง n ครั้ง

$$
C_n=\sum_{i=1}^{n}R_i,\qquad
\mathbb E[R_i]=0,\qquad
\mathbb E[R_i^2]=1,\qquad
\mathbb E[R_iR_j]=0\quad(i\ne j).
$$

ก่อนเริ่มทดลอง E[Cₙ] = 0 และ Var(Cₙ) = n แต่เมื่อรู้ยอดปัจจุบัน Cₘ แล้ว ค่าคาดหมายของยอดในอนาคตคือ **E[Cₙ | Cₘ] = Cₘ เมื่อ n ≥ m** ประวัติที่รู้เปลี่ยนคำตอบของยอดรวม แม้จะไม่เปลี่ยนโอกาสหัวหรือก้อยของครั้งถัดไป

ถ้าบีบ n ครั้งให้อยู่ในเวลารวม T และลดขนาดแต่ละ step เป็น ±√(T/n) จะมี

$$
\Delta t=\frac{T}{n},\qquad
\Delta W=\pm\sqrt{\Delta t},\qquad
\operatorname{Var}\!\left(\sum_{i=1}^{n}\Delta W_i\right)=T.
$$

การเลือก **ขนาด step ตาม √Δt** ทำให้ความสุ่มไม่หายไปเมื่อแบ่งเวลาละเอียดขึ้น ลิมิตของการเดินสุ่มที่ปรับสเกลเหมาะสมนี้ให้ Brownian motion ส่วนในแบบจำลอง Brownian โดยตรง step ในช่วงเวลาที่มีขนาดจำกัดมีการแจกแจง

$$
\Delta W_i=W_{t_{i+1}}-W_{t_i}
\sim\mathcal N(0,\Delta t),
\qquad
\Delta W_i=\sqrt{\Delta t}\,Z_i,\quad Z_i\overset{\text{iid}}{\sim}\mathcal N(0,1).
$$

Brownian motion มีเส้นทางต่อเนื่อง step ในช่วงเวลาที่ไม่ทับกันเป็นอิสระ และ SD ของ step เท่ากับ √Δt สัญลักษณ์ **dW** ใช้ในสมการเชิงสุ่ม ไม่ใช่อนุพันธ์ธรรมดาที่เรานำไปหาร dt แล้วได้ความเร็วของเส้นทางที่เรียบ

ถ้าเวลาเป็นปี W มีหน่วย √ปี จึงต้องดูหน่วยของสัมประสิทธิ์ที่คูณ dW ด้วยเสมอ

In [2]:
T, N, seed = 1., 64, 2510401
dt = T/N
rng = random.Random(seed)
coin_steps = [math.sqrt(dt)*(1 if rng.random()<.5 else -1) for _ in range(N)]
brownian_steps = [math.sqrt(dt)*rng.gauss(0,1) for _ in range(N)]
coin_qv = sum(dw*dw for dw in coin_steps)
brownian_qv = sum(dw*dw for dw in brownian_steps)
close(coin_qv,T)
print(f"T={T}, N={N}, seed={seed}")
print(f"Fair-coin QV={coin_qv:.10f} exactly at this finite grid; Gaussian-increment QV={brownian_qv:.10f}")
print(f"One Gaussian increment squared={brownian_steps[0]**2:.10f}; dt={dt:.10f}: these are not required to agree.")

T=1.0, N=64, seed=2510401
Fair-coin QV=1.0000000000 exactly at this finite grid; Gaussian-increment QV=1.3769679871
One Gaussian increment squared=0.0149580454; dt=0.0156250000: these are not required to agree.


## 2. step เล็กลง แต่ผลรวมกำลังสองยังเหลืออยู่

ให้แบ่งช่วง [0,T] เป็น n ช่วงเท่ากัน แล้ววัด **[quadratic variation](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#quadratic-variation)** บนตารางนี้ด้วย

$$
Q_n=\sum_{i=0}^{n-1}(\Delta W_i)^2.
$$

ต้องแยกสองกรณีที่ดูคล้ายกันออกจากกัน

| step ที่ใช้ | กำลังสองของ step เดียว | ผลรวม Qₙ |
|---|---|---|
| เหรียญ ±√Δt | เท่ากับ Δt ทุกครั้ง | เท่ากับ T พอดี แม้ n ยังจำกัด |
| step Gaussian ของ Brownian | Δt Zᵢ² ยังเป็นตัวแปรสุ่ม | ยังสุ่มที่ n จำกัด แต่เข้าใกล้ T ในลิมิต |

สำหรับ Gaussian increments อิสระ เราใช้ E[Z²] = 1 และ E[Z⁴] = 3 จึงได้

$$
\mathbb E[Q_n]=T,\qquad
\mathbb E[(Q_n-T)^2]
=\operatorname{Var}(Q_n)
=\frac{2T^2}{n}\longrightarrow0.
$$

นี่คือการลู่เข้าแบบ **[mean square](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#mean-square-convergence)**: ค่าเฉลี่ยของความคลาดเคลื่อนยกกำลังสองลดสู่ศูนย์ ไม่ได้หมายความว่าทุกเส้นทางต้องเข้าใกล้ T มากขึ้นในทุกครั้งที่เพิ่ม n

ผลนี้ย่อเป็นกฎจำของ stochastic calculus ว่า

$$
(dW)^2=dt,\qquad dt\,dW=0,\qquad (dt)^2=0.
$$

**กฎนี้เป็นสัญลักษณ์ของการคำนวณในลิมิต ไม่ใช่สมการของ step Gaussian แต่ละ step** ที่ Δt ยังไม่เป็นศูนย์ เราต้องใช้ (ΔW)² = Δt Z² ซึ่งไม่เท่ากับ Δt เสมอ

เทียบกับเส้นทางเรียบ เช่น x(t) = t จะได้ผลรวมกำลังสอง n(T/n)² = T²/n → 0 ความแตกต่างตรงนี้ทำให้แคลคูลัสของ Brownian ต้องเก็บพจน์อันดับสองไว้

In [3]:
T, repetitions, seed = 1., 2000, 2510402
rng = random.Random(seed)
print(f"Monte Carlo QV check: {repetitions} independent paths per N; seed={seed}")
print("N     empirical E[Q_N]   empirical MSE   theoretical MSE=2*T^2/N")
for N in [8,32,128]:
    dt = T/N
    qvs = [dt*sum(rng.gauss(0,1)**2 for _ in range(N)) for _ in range(repetitions)]
    mean = sum(qvs)/repetitions
    mse = sum((q-T)**2 for q in qvs)/repetitions
    theoretical = 2*T*T/N
    assert abs(mse/theoretical-1)<.18  # Sampling tolerance, not exact equality.
    assert abs(mean-T)<.06
    print(f"{N:3d} {mean:19.9f} {mse:15.9f} {theoretical:23.9f}")
print("The L2 result is E[(Q_N-T)^2]=2*T^2/N; Monte Carlo illustrates it with sampling error.")

Monte Carlo QV check: 2000 independent paths per N; seed=2510402
N     empirical E[Q_N]   empirical MSE   theoretical MSE=2*T^2/N
  8         0.991618843     0.258993654             0.250000000
 32         1.002301281     0.064656610             0.062500000
128         1.001424700     0.016271402             0.015625000
The L2 result is E[(Q_N-T)^2]=2*T^2/N; Monte Carlo illustrates it with sampling error.


## 3. ลองยกกำลังสอง แล้วดูสิ่งที่กฎลูกโซ่ธรรมดาทำหาย

เริ่มด้วยเอกลักษณ์พีชคณิตที่ถูกต้องสำหรับทุก step

$$
W_{t_{i+1}}^2-W_{t_i}^2
=2W_{t_i}\Delta W_i+(\Delta W_i)^2.
$$

รวมทุก step และใช้ W₀ = 0 จะได้

$$
W_T^2=2\sum_{i=0}^{n-1}W_{t_i}\Delta W_i+Q_n.
$$

ผลรวมตรงกลางใช้ค่า W **ก่อน** รับช็อกของช่วงถัดไป นี่เป็นจุดเริ่มต้นของ **[Itô integral](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#ito-integral)** ซึ่งใช้ข้อมูลที่มีอยู่แล้ว ไม่เอาค่าในอนาคตมาย้อนกำหนดน้ำหนักของ step ปัจจุบัน

เมื่อแบ่งเวลาละเอียดขึ้น ผลรวมฝั่งซ้ายของแต่ละช่วงให้

$$
\int_0^T W_t\,dW_t=\frac{W_T^2-T}{2}.
$$

จึงได้กฎการเปลี่ยนแปลง

$$
\boxed{d(W_t^2)=2W_t\,dW_t+dt.}
$$

ถ้าใช้กฎลูกโซ่ธรรมดาแล้วเขียนแค่ 2W dW เราจะทำ **dt** หายไป แต่ E[W_T²] = T จึงเห็นได้ว่าพจน์ที่หายมีความหมายจริง

สำหรับ integrand ที่ไม่มองอนาคตและมี E[∫₀ᵀ Hₜ² dt] &lt; ∞ จะมี E[∫₀ᵀ Hₜ dWₜ] = 0 ไม่ควรสรุปว่าปริพันธ์เชิงสุ่มทุกแบบมีค่าคาดหมายศูนย์โดยไม่ดู integrand อ่านนิยามเพิ่มเติมใน [Miranda Holmes-Cerfon, Lecture 7: Stochastic Integration](https://personal.math.ubc.ca/~holmescerfon/teaching/asa22/handout-Lecture7_2022.pdf)

### ทดลอง: ดูพจน์ที่สะสมจากกำลังสองของ step

ห้องทดลองใช้ Gaussian increments ของ Brownian path เดียวกันบนตารางละเอียด แล้วรวม step ติดกันเมื่อเลือก n ที่เล็กลง จึงเปรียบเทียบความละเอียดโดยไม่เปลี่ยนเส้นทางสุ่มพื้นฐาน

ลองเพิ่ม n แล้วเปลี่ยน seed ผลต่างของ Qₙ กับ T อาจขึ้นหรือลงในเส้นทางใดเส้นทางหนึ่ง แต่ค่าคลาดเคลื่อนแบบ RMS ตามทฤษฎีคือ T√(2/n) และผลรวมแบบ Itô บนตารางต้องตรงกับ **(W_T² − Qₙ)/2** ก่อนพิจารณาลิมิต

In [4]:
W, left_sum = 0., 0.
for dw in brownian_steps:
    left_sum += W*dw  # Use the value known BEFORE the increment.
    W += dw
QN = sum(dw*dw for dw in brownian_steps)
finite_identity = (W*W-QN)/2
limit_integral = (W*W-1)/2  # T=1 for the Brownian grid above.
close(left_sum,finite_identity)
close(left_sum-limit_integral,(1-QN)/2)
print(f"W_T={W:.10f}; Q_N={QN:.10f}")
print(f"Finite left sum={left_sum:.10f}; (W_T^2-Q_N)/2={finite_identity:.10f}")
print(f"Ito integral for this endpoint=(W_T^2-T)/2={limit_integral:.10f}")
print(f"Finite-grid error={(left_sum-limit_integral):.10f}=(T-Q_N)/2; ordinary-chain-rule value misses T/2.")

W_T=0.2505957946; Q_N=1.3769679871
Finite left sum=-0.6570848674; (W_T^2-Q_N)/2=-0.6570848674
Ito integral for this endpoint=(W_T^2-T)/2=-0.4686008739
Finite-grid error=-0.1884839936=(T-Q_N)/2; ordinary-chain-rule value misses T/2.


## 4. Itô’s lemma: อ่าน drift และช็อกของฟังก์ชันใหม่

ให้ตัวแปร Y ตาม [stochastic differential equation](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#stochastic-differential-equation)

$$
dY_t=a(Y_t,t)\,dt+b(Y_t,t)\,dW_t.
$$

a เป็น drift ในหน่วย Y ต่อเวลา ส่วน b เป็นขนาดช็อกในหน่วย Y ต่อ √เวลา ค่า a(Yₜ,t) อาจสุ่มเพราะขึ้นกับสถานะ Yₜ คำว่า “ส่วนที่คาดการณ์ได้” จึงหมายถึงรู้สัมประสิทธิ์เมื่อกำหนดสถานะปัจจุบันแล้ว ไม่ได้หมายความว่าเส้นทางของ drift ทั้งหมดไม่สุ่ม

สำหรับฟังก์ชัน F(y,t) ที่หาอนุพันธ์ตามเวลาได้หนึ่งครั้งและตาม y ได้สองครั้ง โดยอนุพันธ์เหล่านี้ต่อเนื่อง หรือเรียกว่า **C¹,²** กฎ Itô ให้

$$
\boxed{
dF(Y_t,t)=
\left(F_t+aF_y+\frac12 b^2F_{yy}\right)dt
+bF_y\,dW_t.
}
$$

อนุพันธ์และสัมประสิทธิ์ทั้งหมดในสูตรประเมินที่ (Yₜ,t) สำหรับกระบวนการที่มีเงื่อนไขความเรียบและการมีอยู่ของคำตอบเหมาะสม

| พจน์ | สิ่งที่ทำให้ F เปลี่ยน |
|---|---|
| Fₜ dt | เวลาเดินไป แม้ตรึงค่า Y |
| aFᵧ dt | drift ของ Y ผ่านความชันของ F |
| ½b²Fᵧᵧ dt | ความผันผวนผ่านความโค้งของ F หรือ Itô correction |
| bFᵧ dW | ช็อกที่เหลืออยู่ใน F |

ที่มาของพจน์ใหม่คือการขยาย Taylor: เมื่อ dY มีขนาดช็อกตาม √dt พจน์ (dY)² จึงมีส่วน b²dt ที่ยังต้องเก็บไว้ หาก F เป็นเส้นตรงจะไม่มี Fᵧᵧ แต่เมื่อ F โค้ง พจน์แก้ไขนี้มักไม่เป็นศูนย์

### เชื่อมกลับไปยังสมการการแพร่

ในบทก่อน dY = √2 c dW จึงมี a = 0 และ b = √2 c เมื่อเลือก F(Y) = Y² จะได้

$$
d(Y^2)=2\sqrt2\,cY\,dW+2c^2dt,
\qquad
\mathbb E[Y_T^2\mid Y_t=y]=y^2+2c^2(T-t).
$$

เทอม **c²Fᵧᵧ** จึงเป็นตัวเดียวกับสัมประสิทธิ์ในสมการ Backward ของบทก่อน ส่วน 2c²(T−t) ตรงกับความแปรปรวนที่เราได้จาก Gaussian

ถ้า F เป็นมูลค่าสัญญา V(S,t) สูตรนี้ช่วยหาการเปลี่ยนแปลงของ V โดยใช้ความชันและความโค้ง แต่ **Itô’s lemma เพียงอย่างเดียวยังไม่กำหนดราคา Option** ต้องมีเงื่อนไข payoff, no arbitrage และแบบจำลองการคิดราคาที่สอดคล้องกันด้วย

In [5]:
def ito_terms(Ft, Fx, Fxx, drift, diffusion):
    return Ft+drift*Fx+.5*diffusion**2*Fxx, diffusion*Fx
# F(Y)=Y^2, dY=sqrt(2)*c*dW links directly to the transition-density chapter.
y, c = 1., 1.
drift_F, diffusion_F = ito_terms(0,2*y,2,0,math.sqrt(2)*c)
close(drift_F,2*c*c)
close(diffusion_F,2*y*math.sqrt(2)*c)
print(f"F(Y)=Y^2 at y=1,c=1: Ito drift={drift_F:.6f}, diffusion={diffusion_F:.6f}")
print("Thus E[Y_T^2 | Y_t=y]=y^2+2*c^2*(T-t), matching the density's variance.")
# A time-dependent function: F(S,t)=exp(-r*t)*S^2 under GBM.
S,t,mu,sigma,r = 100.,.5,.1,.2,.03
F = math.exp(-r*t)*S*S
drift_F,diffusion_F = ito_terms(-r*F,2*F/S,2*F/S**2,mu*S,sigma*S)
close(drift_F,(2*mu+sigma*sigma-r)*F); close(diffusion_F,2*sigma*F)
print(f"Discounted square: drift={drift_F:.9f}, diffusion={diffusion_F:.9f}; includes both F_t and the Ito correction.")

F(Y)=Y^2 at y=1,c=1: Ito drift=2.000000, diffusion=2.828427
Thus E[Y_T^2 | Y_t=y]=y^2+2*c^2*(T-t), matching the density's variance.
Discounted square: drift=2068.735073166, diffusion=3940.447758412; includes both F_t and the Ito correction.


## 5. ใช้กับ GBM: ทำไม log ของราคาจึงมี −½σ²

สมมติราคาตาม [Geometric Brownian Motion](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#geometric-brownian-motion) ที่มี μ,σ คงที่ และ S₀ &gt; 0

$$
dS_t=\mu S_t\,dt+\sigma S_t\,dW_t.
$$

เมื่อเวลาเป็นปี μ มีหน่วยต่อปี และ σ มีหน่วยต่อ √ปี ในตัวอย่างจะใช้ μ = 0.10 และ σ = 0.20 ค่า μ เป็น drift ของผลตอบแทนแบบสัดส่วนในโมเดล ไม่ใช่ผลตอบแทนที่เส้นทางจริงต้องได้

เลือก F(S) = log S จะมี F_S = 1/S และ F_SS = −1/S² จึงได้

$$
d\log S_t
=\left(\mu-\frac12\sigma^2\right)dt+\sigma\,dW_t.
$$

log เป็นฟังก์ชันเว้า อนุพันธ์อันดับสองจึงติดลบและทำให้ drift ของ log ต่ำกว่า μ เมื่อ σ ไม่เป็นศูนย์ นี่คือผลของ Itô correction ไม่ใช่ค่าธรรมเนียมหรือการหักผลตอบแทนตามใจ

อินทิเกรตจาก 0 ถึง t โดยใช้ W₀ = 0:

$$
\log\frac{S_t}{S_0}
=\left(\mu-\frac12\sigma^2\right)t+\sigma W_t,
\qquad
\boxed{S_t=S_0\exp\!\left[\left(\mu-\frac12\sigma^2\right)t+\sigma W_t\right].}
$$

คำตอบนี้เป็นบวกทุกเวลาจำกัดเมื่อ S₀ &gt; 0 และทำให้ **log return เป็น Normal ส่วนราคาเป็น Lognormal** ไม่ใช่เพราะ simple return ถูกสมมติให้เป็น Normal แล้วราคาจะเป็น Lognormal โดยทันที

สำหรับ σ &gt; 0 และ t &gt; 0:

$$
\log\frac{S_t}{S_0}
\sim\mathcal N\!\left(\left(\mu-\frac12\sigma^2\right)t,\sigma^2t\right).
$$

เมื่อ S₀ = 100, μ = 0.10, σ = 0.20 และ t = 1 ปี จะมี

| สิ่งที่วัด | สูตร | ค่า |
|---|---|---:|
| ค่าคาดหมายของราคา | S₀e^(μt) | 110.5171 |
| มัธยฐานของราคา | S₀e^((μ−σ²/2)t) | 108.3287 |
| ค่าคาดหมายของ log return | (μ−σ²/2)t | 0.08 |
| SD ของ log return | σ√t | 0.20 |

ค่าเฉลี่ยกับมัธยฐานต่างกันเพราะการแจกแจงราคาเบ้ ค่า 108.3287 จึงไม่ใช่ E[S₁] และไม่มีค่าใดเป็นการทำนายราคาจริง ณ สิ้นปี

**อีกตัวอย่าง: ราคายกกำลังสอง**

เลือก F(S) = S² มี F_S = 2S และ F_SS = 2 ดังนั้น

$$
d(S_t^2)=(2\mu+\sigma^2)S_t^2\,dt+2\sigma S_t^2\,dW_t.
$$

คราวนี้ฟังก์ชันนูนทำให้ Itô correction เป็นบวก สังเกตเครื่องหมายเทียบกับกรณี log S แล้วจะเห็นบทบาทของความโค้งโดยไม่ต้องท่องสูตรแยกกัน

In [6]:
S0,mu,sigma,T = 100.,.1,.2,1.
log_drift,log_diffusion = ito_terms(0,1/S0,-1/S0**2,mu*S0,sigma*S0)
close(log_drift,.08); close(log_diffusion,.2)
mean = S0*math.exp(mu*T)
median = S0*math.exp((mu-.5*sigma*sigma)*T)
close(mean,110.51709180756477); close(median,108.32870676749586)
print(f"Log-return mean={log_drift*T:.6f}, variance={sigma*sigma*T:.6f}")
print(f"GBM S0=100, mu=.1, sigma=.2, T=1: mean={mean:.9f}, median={median:.9f}")
for WT in [-3.,0.,2.]:
    terminal = S0*math.exp((mu-.5*sigma*sigma)*T+sigma*WT)
    assert terminal > 0
    close(math.log(terminal/S0),(mu-.5*sigma*sigma)*T+sigma*WT)
    print(f"W_T={WT:+.1f}: exact terminal S={terminal:.9f}")

Log-return mean=0.080000, variance=0.040000
GBM S0=100, mu=.1, sigma=.2, T=1: mean=110.517091808, median=108.328706767
W_T=-3.0: exact terminal S=59.452054797
W_T=+0.0: exact terminal S=108.328706767
W_T=+2.0: exact terminal S=161.607440219


## 6. ใช้กับ mean reversion: กลับเข้าหาค่าเฉลี่ยได้โดยยังมีช็อก

GBM ให้ขนาดช็อกโตตามราคา แต่อีกกลุ่มหนึ่งของแบบจำลองมีแรงดึงกลับเข้าหาค่ากลางหรือ [mean reversion](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#mean-reversion) ตัวอย่างคือ **[Ornstein–Uhlenbeck process](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#ornstein-uhlenbeck)** หรือ **Vasicek model** เมื่อใช้กับอัตราดอกเบี้ยระยะสั้น

$$
dr_t=\kappa(\theta-r_t)\,dt+\sigma_r\,dW_t,
\qquad \kappa>0.
$$

θ เป็นระดับระยะยาว κ เป็นความเร็วในการกลับเข้าหา θ และ σᵣ เป็นขนาดช็อกของระดับ r สมมติทั้งสามเป็นค่าคงที่สำหรับคำตอบในหัวข้อนี้ ในบทนี้เปลี่ยน γ และ r̄ ของต้นฉบับเป็น κ และ θ เพื่อให้อ่านง่าย

เมื่อ r ต่ำกว่า θ พจน์ drift เป็นบวก เมื่อ r สูงกว่า θ drift เป็นลบ แต่ช็อกยังอาจผลักให้เดินออกห่างได้ **mean reversion เป็นแรงดึงใน drift ไม่ใช่คำสัญญาว่าทุก step ต้องเข้าใกล้ค่าเฉลี่ย**

ตั้ง uₜ = rₜ − θ จะได้ du = −κu dt + σᵣ dW แล้วใช้ integrating factor e^(κt):

$$
d(e^{\kappa t}u_t)=\sigma_r e^{\kappa t}\,dW_t.
$$

จึงได้คำตอบ

$$
r_t=\theta+(r_0-\theta)e^{-\kappa t}
+\sigma_r\int_0^t e^{-\kappa(t-s)}\,dW_s.
$$

สำหรับ r₀ ที่กำหนดแน่นอนและ σᵣ &gt; 0 คำตอบมีการแจกแจง Normal โดย

$$
\begin{aligned}
\mathbb E[r_t\mid r_0]&=\theta+(r_0-\theta)e^{-\kappa t},\\
\operatorname{Var}(r_t\mid r_0)&=\frac{\sigma_r^2}{2\kappa}\left(1-e^{-2\kappa t}\right).
\end{aligned}
$$

เมื่อเวลายาวขึ้น อิทธิพลของ r₀ ลดลงและการแจกแจงเข้าใกล้ **[stationary distribution](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#stationary-distribution)**

$$
r_\infty\sim\mathcal N\!\left(\theta,\frac{\sigma_r^2}{2\kappa}\right).
$$

นี่หมายถึงการแจกแจงระยะยาวคงรูป ไม่ได้หมายความว่าเส้นทางแต่ละเส้นหยุดสุ่ม หากเริ่มจาก r₀ คงที่ กระบวนการยังไม่อยู่ใน stationary distribution ตั้งแต่เวลา 0

ตัวอย่างสมมติให้ r₀ = 5%, θ = 3%, κ = 1.5 ต่อปี และ σᵣ = 0.01 ต่อ √ปี หรือ 1 จุดเปอร์เซ็นต์ต่อ √ปี หลังหนึ่งปี ค่าคาดหมายประมาณ **3.4463%** และ SD ประมาณ **0.5628 จุดเปอร์เซ็นต์** ส่วน SD ระยะยาวประมาณ **0.5774 จุดเปอร์เซ็นต์**

OU/Vasicek มีหาง Gaussian บนเส้นจำนวนจริง จึงยอมให้อัตราดอกเบี้ยติดลบได้ การเลือกใช้ต้องพิจารณาว่าสมบัตินี้เหมาะกับสิ่งที่กำลังจำลองหรือไม่

In [7]:
def ou_moments(initial,theta,kappa,sigma,tau):
    if kappa <= 0 or sigma < 0 or tau < 0:
        raise ValueError("Require kappa>0, sigma>=0 and tau>=0")
    mean = theta+(initial-theta)*math.exp(-kappa*tau)
    variance = sigma*sigma*(-math.expm1(-2*kappa*tau))/(2*kappa)
    return mean,variance
r0,theta,kappa,sigma_r,T = .05,.03,1.5,.01,1.
mean,variance = ou_moments(r0,theta,kappa,sigma_r,T)
sd,stationary_sd = math.sqrt(variance),sigma_r/math.sqrt(2*kappa)
close(mean,.034462603202968595); close(sd,.005627944952443819)
close(stationary_sd,.005773502691896258)
long_mean,long_variance = ou_moments(r0,theta,kappa,sigma_r,100.)
close(long_mean,theta); close(long_variance,stationary_sd**2)
print(f"OU/Vasicek T=1: mean={mean:.12f}, SD={sd:.12f}; stationary SD={stationary_sd:.12f}")
print("Rate units are decimals: SD .005627945 is approximately .5627945 percentage points.")

OU/Vasicek T=1: mean=0.034462603203, SD=0.005627944952; stationary SD=0.005773502692
Rate units are decimals: SD .005627945 is approximately .5627945 percentage points.


## 7. SDE หนึ่งสมการ เชื่อมกับความหนาแน่นสองมุมมอง

กลับไปที่ตัวแปรทั่วไป dY = a(Y,t)dt + b(Y,t)dW และใช้สัญลักษณ์เดิมจากบทก่อน **k(y,t;z,T)** เมื่อมี transition density และความเรียบเพียงพอ สมการ Forward คือ

$$
\boxed{
\frac{\partial k}{\partial T}
=-\frac{\partial}{\partial z}\big[a(z,T)k\big]
+\frac12\frac{\partial^2}{\partial z^2}\big[b(z,T)^2k\big].
}
$$

Forward ตรึงจุดเริ่มต้น y,t แล้วติดตามการแจกแจงปลายทาง **สัมประสิทธิ์อยู่ภายในอนุพันธ์** เพราะอาจเปลี่ยนตามตำแหน่ง z ส่วน Backward ตรึงปลายทาง z,T และทำงานกับจุดเริ่มต้น:

$$
\boxed{
\frac{\partial k}{\partial t}
+a(y,t)\frac{\partial k}{\partial y}
+\frac12b(y,t)^2\frac{\partial^2k}{\partial y^2}=0.
}
$$

ถ้า a = 0 และ b = √2 c คงที่ ทั้งคู่จะกลับเป็นสมการในบท Transition Density ทันที แต่เมื่อสัมประสิทธิ์ขึ้นกับสถานะ จะสลับตัวแปรและเปลี่ยนเครื่องหมายอย่างเดียวไม่ได้

### จาก log ของ GBM กลับเป็นความหนาแน่นของราคา

เพราะ log return เป็น Normal เราจึงแปลงกลับเป็นความหนาแน่นของราคาปลายทาง z &gt; 0 ได้ สำหรับ τ = T − t &gt; 0, ราคาเริ่มต้น s &gt; 0 และ σ &gt; 0:

$$
k(s,t;z,T)
=\frac{1}{z\sigma\sqrt{2\pi\tau}}
\exp\!\left[
-\frac{\left(\log(z/s)-(\mu-\sigma^2/2)\tau\right)^2}{2\sigma^2\tau}
\right].
$$

ตัวคูณ **1/z** มาจากการเปลี่ยนตัวแปร log z กลับเป็น z จะลืมพจน์นี้ไม่ได้ เพราะพื้นที่ใต้ความหนาแน่นราคาต้องยังเป็น 1

สำหรับ OU, drift ดึงมวลกลับเข้าหา θ ขณะที่ diffusion กระจายมวลออก จึงมีการแจกแจงระยะยาวที่สมดุลกันได้ การมีสัมประสิทธิ์ที่ไม่ขึ้นกับเวลาเพียงอย่างเดียวไม่ได้รับประกันว่าจะมี stationary distribution สำหรับทุกแบบจำลอง

สมการเหล่านี้ต้องใช้ร่วมกับเงื่อนไขเริ่มต้น ขอบเขต และกฎความน่าจะเป็นที่เลือก การหาความหนาแน่นภายใต้แบบจำลองยังไม่เท่ากับการกำหนดราคาสัญญาภายใต้ risk-neutral measure อ่านเรื่องตัวดำเนินการและเงื่อนไขต่อได้ใน [Lecture 10: Forward and Backward Equations for SDEs](https://personal.math.ubc.ca/~holmescerfon/teaching/asa22/handout-Lecture10_2022.pdf)

In [8]:
# Verify the general forward/backward equations using a nonconstant OU drift.
kappa,theta,sigma_r = 1.5,.03,.01
def kernel(y,t,z,T):
    mean,variance = ou_moments(y,theta,kappa,sigma_r,T-t)
    return math.exp(-(z-mean)**2/(2*variance))/math.sqrt(2*math.pi*variance)
y,t,z,T = .05,0.,.037,1.
et,ex = 1e-5,1e-6
p = kernel(y,t,z,T)
p_T = (kernel(y,t,z,T+et)-kernel(y,t,z,T-et))/(2*et)
p_t = (kernel(y,t+et,z,T)-kernel(y,t-et,z,T))/(2*et)
p_y = (kernel(y+ex,t,z,T)-kernel(y-ex,t,z,T))/(2*ex)
p_yy = (kernel(y+ex,t,z,T)-2*p+kernel(y-ex,t,z,T))/ex**2
p_zz = (kernel(y,t,z+ex,T)-2*p+kernel(y,t,z-ex,T))/ex**2
divergence = (kappa*(theta-z-ex)*kernel(y,t,z+ex,T)-kappa*(theta-z+ex)*kernel(y,t,z-ex,T))/(2*ex)
forward = p_T+divergence-.5*sigma_r*sigma_r*p_zz
backward = p_t+kappa*(theta-y)*p_y+.5*sigma_r*sigma_r*p_yy
close(forward,0,2e-5); close(backward,0,2e-5)
print(f"OU finite-difference residual: forward={forward:.3e}, backward={backward:.3e}")
print("Forward differentiates the drift*density product; backward multiplies a derivative by the initial-state drift.")

OU finite-difference residual: forward=1.861e-07, backward=5.333e-07
Forward differentiates the drift*density product; backward multiplies a derivative by the initial-state drift.


## 8. จากสมการต่อเนื่อง สู่ขั้นตอนที่คอมพิวเตอร์คำนวณได้

วิธี **[Euler–Maruyama](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#euler-maruyama)** ประมาณการเปลี่ยนแปลงโดยประเมินสัมประสิทธิ์ที่ต้นช่วง แล้วใช้ช็อกใหม่ในแต่ละ step:

$$
Y_{i+1}=Y_i+a(Y_i,t_i)\Delta t
+b(Y_i,t_i)\sqrt{\Delta t}\,Z_i,
\qquad Z_i\overset{\text{iid}}{\sim}\mathcal N(0,1).
$$

ช็อกในสูตรนี้คูณด้วย √Δt หากใช้ Δt แทน ความแปรปรวนรวมจะยุบลงเมื่อแบ่งเวลาละเอียดขึ้นและได้คนละแบบจำลอง

สำหรับ GBM สูตร Euler เป็น

$$
S_{i+1}^{\mathrm{Euler}}
=S_i^{\mathrm{Euler}}\left(1+\mu\Delta t+\sigma\sqrt{\Delta t}\,Z_i\right).
$$

ใช้ค่าจากตัวอย่างในเอกสาร: S₀ = 100, μ = 0.1, σ = 0.2, Δt = 0.01 ปี และช็อกสองค่า 0.12, −0.25 จะได้

| step | การคำนวณ | ราคาหลังจบ step |
|---|---|---:|
| 1 | 100 × (1 + 0.1×0.01 + 0.2×0.1×0.12) | 100.34 |
| 2 | 100.34 × (1 + 0.1×0.01 + 0.2×0.1×(−0.25)) | 99.93864 |

ต้นฉบับปัดค่าที่สองเป็น 99.94 สำหรับการแสดงผล แต่การคำนวณต่อควรเก็บความละเอียดไว้

### เมื่อมีคำตอบ exact ให้ใช้เป็นตัวเทียบ

GBM ที่ μ,σ คงที่มีสูตรเปลี่ยนสถานะที่แน่นอนบนจุดเวลาที่เลือก:

$$
S_{i+1}^{\mathrm{exact}}
=S_i^{\mathrm{exact}}
\exp\!\left[\left(\mu-\frac12\sigma^2\right)\Delta t
+\sigma\sqrt{\Delta t}\,Z_i\right].
$$

Euler กับ exact ควรใช้ **ช็อกเดียวกัน** เพื่อดูความคลาดเคลื่อนจากวิธีคำนวณ หากสุ่มคนละเส้นทาง ผลต่างจะปนทั้งความสุ่มและความคลาดเคลื่อนของวิธี

คำว่า exact ในที่นี้หมายถึงคำตอบของ **GBM ที่สมมติไว้ ณ จุดเวลาที่คำนวณ** เส้นเชื่อมจุดบนกราฟเป็นภาพช่วยอ่าน ไม่ใช่การแสดงรายละเอียดทุก step ของเส้นทางต่อเนื่อง และคำตอบ exact ของแบบจำลองไม่ได้รับประกันความถูกต้องของแบบจำลองต่อโลกจริง

Euler อาจให้ราคาศูนย์หรือติดลบถ้าตัวคูณ 1 + μΔt + σ√Δt Z ไม่เป็นบวก แม้คำตอบ GBM จริงจะเป็นบวก ตัวอย่างสมมติ μ = 0.1, σ = 1, Δt = 1 และ Z = −2 ให้ตัวคูณ Euler = −0.9 แต่ตัวคูณ exact = e^(−2.4) &gt; 0 การตัดค่าติดลบให้เป็นศูนย์จะเปลี่ยนวิธีและการแจกแจง จึงไม่ควรทำโดยไม่อธิบาย

เมื่อย่อ Δt ภายใต้เงื่อนไขที่เหมาะสม Euler–Maruyama ลู่เข้าในความหมายทางความน่าจะเป็นที่กำหนด แต่ **error ของเส้นทางเดียวไม่จำเป็นต้องลดทุกครั้ง** การตรวจความแม่นควรใช้หลายเส้นทางร่วมกับช็อกที่จับคู่กัน

| แหล่งความคลาดเคลื่อน | เกิดจากอะไร | สิ่งที่ช่วยลด |
|---|---|---|
| [การแบ่งเวลา](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#discretization-error) | แทน SDE ด้วย step ที่ยังมีขนาดจำกัด | ย่อ Δt หรือเลือกวิธีเหมาะสม |
| Monte Carlo sampling error | ประมาณค่าคาดหมายด้วยเส้นทางจำนวนจำกัด | เพิ่มจำนวนเส้นทางและวัด standard error |
| Model error | สมมติฐานไม่ตรงกับสิ่งที่ต้องการอธิบาย | ตรวจแบบจำลองกับข้อมูลและโจทย์จริง |

In [9]:
S0,mu,sigma,dt = 100.,.1,.2,.01
S = S0
for Z,expected in zip([.12,-.25],[100.34,99.93864]):
    S *= 1+mu*dt+sigma*math.sqrt(dt)*Z
    close(S,expected)
    print(f"Euler with Z={Z:+.2f}: S={S:.8f}")
T,fine_N,seed = 1.,1024,2510403
rng = random.Random(seed)
fine = [math.sqrt(T/fine_N)*rng.gauss(0,1) for _ in range(fine_N)]
WT = sum(fine)
exact = S0*math.exp((mu-.5*sigma*sigma)*T+sigma*WT)
print(f"Same Brownian path at every resolution: seed={seed}, exact terminal={exact:.9f}")
for N in [16,64,256,1024]:
    group = fine_N//N
    increments = [sum(fine[j:j+group]) for j in range(0,fine_N,group)]
    close(sum(increments),WT)
    euler,exact_steps = S0,S0
    for dw in increments:
        euler *= 1+mu*T/N+sigma*dw
        exact_steps *= math.exp((mu-.5*sigma*sigma)*T/N+sigma*dw)
    close(exact_steps,exact)
    print(f"N={N:4d}: Euler={euler:.9f}, absolute error={abs(euler-exact):.9f}")
print("Pathwise discretization errors need not decrease monotonically; this is one coupled path, not a convergence proof.")

Euler with Z=+0.12: S=100.34000000
Euler with Z=-0.25: S=99.93864000
Same Brownian path at every resolution: seed=2510403, exact terminal=114.760254935
N=  16: Euler=114.601854827, absolute error=0.158400108
N=  64: Euler=115.015911875, absolute error=0.255656940
N= 256: Euler=114.834058056, absolute error=0.073803121
N=1024: Euler=114.752960733, absolute error=0.007294202
Pathwise discretization errors need not decrease monotonically; this is one coupled path, not a convergence proof.


## 9. ตัวสุ่มที่มีค่าเฉลี่ยและ SD ถูก ยังอาจมีหางผิด

สองเอกสารพูดถึงการสร้างตัวสุ่ม Normal สำหรับจำลอง SDE แนวคิด inverse transform คือ ถ้า U กระจายสม่ำเสมอบน (0,1) แล้ว **Z = Φ⁻¹(U)** จะมีการแจกแจง Standard Normal โดย Φ เป็น CDF ของ Normal

อีกวิธีในเอกสารคือผลรวม uniform 12 ตัวลบ 6 ซึ่งมีค่าเฉลี่ย 0 และความแปรปรวน 1 จริง แต่มีค่าจำกัดอยู่ในช่วง [−6,6] จึงเป็นเพียง **Normal approximation** และไม่สามารถแทนหาง Normal ได้ทุกระดับ

ห้องทดลองในบทใช้ตัวสร้างเลขสุ่มที่ระบุ seed และแปลงเป็น Normal เพื่อให้ทดลองซ้ำได้ ส่วน Notebook ระบุวิธีและ seed ของตนเอง การใช้ seed เดียวกันต่างภาษาไม่ได้รับประกันว่าจะได้เลขชุดเดียวกัน ถ้าอัลกอริทึมสุ่มต่างกัน

การตรวจตัวสุ่มต้องดูทั้งชนิดการแจกแจงและหาง ควบคู่กับค่าเฉลี่ยและ SD

## 10. จากช็อกเดียว สู่ช็อกสองตัวที่สัมพันธ์กัน

ถ้าจำลองสองสินทรัพย์โดยให้ช็อกเป็นอิสระเสมอ เรากำลังกำหนดโครงสร้างความเสี่ยงบางอย่างไว้แล้ว การสร้าง [correlated increments](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#correlated-increments) เริ่มจากช็อก Standard Normal ที่มี correlation ρ โดยใช้ Z₁,Z₂ ซึ่งเป็น **อิสระต่อกัน** และต่างก็เป็น Standard Normal:

$$
\phi_1=Z_1,\qquad
\phi_2=\rho Z_1+\sqrt{1-\rho^2}\,Z_2,
\qquad -1\le\rho\le1.
$$

จะได้ E[φ₁] = E[φ₂] = 0, Var(φ₁) = Var(φ₂) = 1 และ

$$
\operatorname{Cov}(\phi_1,\phi_2)
=\mathbb E[\phi_1\phi_2]=\rho.
$$

ตัวอย่าง ρ = −0.6, Z₁ = 1, Z₂ = −0.5 ให้ φ₁ = 1 และ φ₂ = −0.6×1 + 0.8×(−0.5) = **−1** นี่เป็นเพียงช็อกหนึ่งคู่ ค่า correlation เป็นคุณสมบัติของการแจกแจงร่วม ต้องดูหลายคู่จึงประมาณจากตัวอย่างได้

เมื่อ ρ = 1 จะมี φ₂ = φ₁ และเมื่อ ρ = −1 จะมี φ₂ = −φ₁ ทุกคู่ ส่วน ρ = 0 ให้ช็อกสองตัวอิสระในโครงสร้าง Gaussian นี้ การเลื่อน ρ ใช้ Z₁,Z₂ ชุดเดิมเพื่อให้เห็นผลของพารามิเตอร์

ในการจำลองตามเวลา ให้สุ่มคู่ Z₁,Z₂ ใหม่อย่างอิสระจากคู่ในช่วงก่อนหน้า เมื่อแปลงเป็น step Brownian ด้วย ΔW₁ = √Δt φ₁ และ ΔW₂ = √Δt φ₂ จะมี Cov(ΔW₁,ΔW₂) = ρΔt หรือเขียนย่อว่า

$$
dW_1\,dW_2=\rho\,dt.
$$

ρ นี้เป็น correlation ของ **ช็อกในช่วงเวลาเดียวกัน** ไม่ใช่ correlation ของระดับราคาที่วัดตามเวลาบนเส้นทางหนึ่งโดยอัตโนมัติ และ sample correlation จากจำนวนคู่จำกัดไม่จำเป็นต้องเท่ากับค่าเป้าหมายพอดี

**เมื่อบวกตัวแปร Normal ต้องดู covariance ด้วย**

ผลรวมเชิงเส้นของตัวแปรที่เป็น jointly Gaussian ยังเป็น Normal หากตัวแปรอิสระและแต่ละตัวเป็น Normal จะเข้าเงื่อนไขนี้ ส่วนการรู้ว่าแต่ละตัวมี marginal Normal อย่างเดียวไม่พอ

$$
\operatorname{Var}\!\left(\sum_i w_iX_i\right)
=\sum_i w_i^2\operatorname{Var}(X_i)
+2\sum_{i<j}w_iw_j\operatorname{Cov}(X_i,X_j).
$$

จะตัดพจน์ covariance ทิ้งได้เมื่อเป็นศูนย์ ในสูตรสร้าง φ₁,φ₂ เราเริ่มจาก Z₁,Z₂ อิสระ แล้วจึงสร้างความสัมพันธ์ขึ้นอย่างควบคุมได้ ข้อสมมตินี้เติมให้ชัดจากสูตรสรุปในเอกสาร

In [10]:
def innovations(z1,z2,rho):
    if not -1 <= rho <= 1:
        raise ValueError("rho must be in [-1,1]")
    return z1,rho*z1+math.sqrt(1-rho*rho)*z2
close(innovations(1,-.5,-.6)[1],-1)
close(innovations(1,-.5,1)[1],1); close(innovations(1,-.5,-1)[1],-1)
seed,count,rho = 2510404,20000,-.6
rng = random.Random(seed)
pairs = [innovations(rng.gauss(0,1),rng.gauss(0,1),rho) for _ in range(count)]
m1,m2 = (sum(p[j] for p in pairs)/count for j in [0,1])
v1,v2 = (sum((p[j]-m)**2 for p in pairs)/count for j,m in [(0,m1),(1,m2)])
cov = sum((a-m1)*(b-m2) for a,b in pairs)/count
correlation = cov/math.sqrt(v1*v2)
assert abs(correlation-rho)<.035 and abs(v1-1)<.05 and abs(v2-1)<.05
print("Fixed example: rho=-.6, Z1=1, Z2=-.5 gives (phi1,phi2)=(1,-1).")
print(f"{count} pairs, seed={seed}: variances=({v1:.6f},{v2:.6f}), sample correlation={correlation:.6f}")
print("These are correlations of innovations, not a claim about a finite sample of asset price levels.")
print("All assertions passed.")

Fixed example: rho=-.6, Z1=1, Z2=-.5 gives (phi1,phi2)=(1,-1).
20000 pairs, seed=2510404: variances=(0.994074,0.994530), sample correlation=-0.595784
These are correlations of innovations, not a claim about a finite sample of asset price levels.
All assertions passed.


## ลองเชื่อมเครื่องมือเข้าด้วยกัน

1. ถ้า Δt = 0.01 และ Z = 2 step ΔW และกำลังสองของ step เป็นเท่าไร กฎ (dW)² = dt บอกว่าค่านี้ต้องเท่ากับ 0.01 หรือไม่
2. ถ้า dY = 0.3dt + 0.4dW และ F(Y) = Y² จงหา dF
3. GBM มี μ = 0.10 และ σ = 0.20 drift ของ log S เป็นเท่าไร และเท่ากับ drift ของผลตอบแทน dS/S หรือไม่
4. ใน OU ถ้า r สูงกว่า θ จะสรุปได้หรือไม่ว่า step ถัดไป r ต้องลด
5. ถ้าจะเปรียบเทียบ Euler กับ exact ควรใช้ช็อกชุดเดียวกันหรือคนละชุด และเพราะเหตุใด
6. ถ้าเพิ่มจำนวนเส้นทาง Monte Carlo อย่างเดียว ความผิดพลาดจาก Δt ที่หยาบและสมมติฐานโมเดลจะหายหรือไม่

**เปิดเฉลยพร้อมเหตุผล**

1. ΔW = √0.01 × 2 = **0.2** และ (ΔW)² = **0.04** กฎเชิง differential ไม่ใช่การบังคับให้กำลังสองของ step ที่มีขนาดจำกัดเท่ากับ Δt
2. F_Y = 2Y, F_YY = 2 จึงได้ **dF = (0.6Y + 0.16)dt + 0.8Y dW** พจน์ 0.16 มาจาก Itô correction
3. **0.08 ต่อปี** เทียบกับ drift 0.10 ต่อปีของผลตอบแทนเชิง differential dS/S จึงไม่เท่ากัน
4. **สรุปไม่ได้** drift เป็นลบ แต่ช็อกอาจเป็นบวกและมีขนาดใหญ่กว่าแรงดึงกลับ
5. **ชุดเดียวกัน** เพื่อแยกความต่างของวิธีคำนวณออกจากความต่างของเส้นทางสุ่ม
6. **ไม่หาย** การเพิ่มเส้นทางลด sampling error แต่ยังต้องตรวจ discretization error และ model error แยกกัน

เครื่องมือแต่ละชิ้นมีหน้าที่ต่อกัน: quadratic variation อธิบายพจน์ที่กฎธรรมดาทิ้งไม่ได้, Itô’s lemma แปลงฟังก์ชันของกระบวนการ, Kolmogorov อธิบายการแจกแจง และการจำลองช่วยคำนวณเมื่อไม่สะดวกใช้คำตอบวิเคราะห์

## อ้างอิงและขอบเขตการเรียบเรียง

- เนื้อหาเรียบเรียงภาษาไทยใหม่จากสองไฟล์ เปลี่ยน X ของ Brownian เป็น W และเปลี่ยน γ,r̄ ของ Vasicek เป็น κ,θ เพื่อรักษาความต่อเนื่องกับบทก่อน ไม่เผยแพร่ PDF หรือภาพสไลด์ต้นฉบับ
- ส่วนที่ขยายให้ชัด ได้แก่ การแยก coin increments จาก Gaussian increments, ความหมายของ (dW)² = dt, เงื่อนไข Itô integral และความเรียบของ F, ค่าเฉลี่ยกับมัธยฐานของ GBM, moments ระหว่างทางของ OU, ความคลาดเคลื่อนสามประเภท และข้อสมมติ jointly Gaussian / covariance ตัวอย่างและห้องทดลองคำนวณขึ้นใหม่ ยกเว้นตัวอย่าง Euler สอง step ที่ระบุว่ามาจากเอกสาร
- Miranda Holmes-Cerfon, <em>Applied Stochastic Analysis</em> (Spring 2022): [Lecture 7 — Stochastic Integration](https://personal.math.ubc.ca/~holmescerfon/teaching/asa22/handout-Lecture7_2022.pdf), [Lecture 8 — Stochastic Differential Equations](https://personal.math.ubc.ca/~holmescerfon/teaching/asa22/handout-Lecture8_2022.pdf), [Lecture 10 — Forward and Backward Equations](https://personal.math.ubc.ca/~holmescerfon/teaching/asa22/handout-Lecture10_2022.pdf) ใช้ตรวจนิยาม เงื่อนไข และการเชื่อม SDE กับสมการความหนาแน่น